# Project 5 — Notebook 18: Business Summary & Recommendations
### Site-Level Risk Profiling · Manpower Analysis · Targeted PM Prioritisation

---

| | |
|---|---|
| **Scope** | NCR (Region 3) · Site risk profiling · Engineer load analysis |
| **Feeds from** | NB16 (Site Risk Profiling) · NB17 (Manpower & NOC Quality) |
| **Audience** | Operations Leadership / Area Heads |

---

### What Project 5 Investigated

| Question | Notebook | Status |
|----------|----------|--------|
| How complex is each site — and does complexity correlate with performance? | NB16 | ✅ |
| Which sites are chronically underperforming across frequency, MTTR, and SLA? | NB16 | ✅ |
| Do Zone 1's extreme MTTR outlier sites have a specific RFO root cause? | NB16 | ✅ |
| Is field engineer load equitably distributed by zone (Field Operations, zone-aligned, ≥100 tickets)? | NB16–17 | ✅ |
| Which non-field groups handle WOs in the field dispatch path? | NB17 | ✅ |
| Does the ROC team account explain San Juan City's data anomalies? | NB17 | ✅ |
| Which engineers carry site loads that create day-off response risk? | NB17 | ✅ |

## 1. Setup

In [1]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline

os.chdir(os.path.join('..', '..'))
if os.path.abspath(os.getcwd()) not in sys.path:
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from config import ZONE_ORDER, ZONE_PALETTE

df      = pd.read_csv('output/cleaned_fault_ticket.csv')
df_zone = df[(df['ZONE'].isin(ZONE_ORDER)) & (df['Priority'] < 4)].copy()
df_fd   = df_zone[df_zone['Resolution_Path'] == 'Field_Dispatch_Restored'].copy()

# Field Operations: WOOwnerGroup normalised category (set by pipeline)
FIELD_ENG_GROUPS = ['Field Operations']

# Field engineer tickets: FD, attributed, Field Operations group only
df_fe_all = df_fd[df_fd['Field_Lead_ID'] != 'Unknown'].copy()
MIN_FE_TICKETS = 100  # must match NB16 and NB17

if 'WOOwnerGroup' in df_fe_all.columns:
    # Step 1 — Field Operations WOs only
    _fe_fo = df_fe_all[df_fe_all['WOOwnerGroup'] == 'Field Operations'].copy()
    # Step 2 — Zone alignment
    _home = (_fe_fo.groupby('Field_Lead_ID')['ZONE']
        .agg(lambda x: x.mode()[0]).rename('Home_Zone').reset_index())
    _fe_fo = _fe_fo.merge(_home, on='Field_Lead_ID', how='left')
    _fe_fo = _fe_fo[_fe_fo['ZONE'] == _fe_fo['Home_Zone']].copy()
    # Step 3 — Activity threshold
    _zc  = _fe_fo.groupby(['ZONE','Field_Lead_ID']).size().reset_index(name='_tc')
    _act = _zc[_zc['_tc'] >= MIN_FE_TICKETS][['ZONE','Field_Lead_ID']]
    df_fe = _fe_fo.merge(_act, on=['ZONE','Field_Lead_ID'], how='inner').copy()
else:
    df_fe = df_fe_all  # fallback if pipeline not yet re-run

ROC_ID  = 'ENG_a2fa1599'
CORE_IP = ['Core Network', 'IP/Network Infra', 'IP Network Infra']
MIN_SITE_TICKETS = 10

# Site complexity
site_ne = (df_zone.groupby('SiteName')
    .agg(NE_Count=('NEType','nunique'),
         Has_Core_IP=('NE_Category', lambda x: x.isin(CORE_IP).any()),
         ZONE=('ZONE', lambda x: x.mode()[0]),
         CITY=('CITY', lambda x: x.mode()[0]))
    .reset_index()
)
def classify(row):
    if row['Has_Core_IP'] or row['NE_Count'] >= 6: return 'High'
    if row['NE_Count'] >= 3: return 'Medium'
    return 'Low'
site_ne['Complexity'] = site_ne.apply(classify, axis=1)

# Site risk scores
date_months = (pd.to_datetime(df_zone['REPORTDATE']).max() -
               pd.to_datetime(df_zone['REPORTDATE']).min()).days / 30.44

site_vol = df_zone.groupby(['ZONE','SiteName']).agg(
    All_Tickets=('SLA_Compliant','count')).reset_index()
site_fd  = df_fd.groupby(['ZONE','SiteName']).agg(
    FD_Tickets=('SLA_Compliant','count'),
    SLA_pct=('SLA_Compliant','mean'),
    Avg_MTTR=('OUTAGEDURATION','mean')).reset_index()
site_fd['SLA_pct'] *= 100
site_fd['Breach_Rate'] = 100 - site_fd['SLA_pct']

site_risk = (site_vol
    .merge(site_fd, on=['ZONE','SiteName'], how='left')
    .merge(site_ne[['SiteName','Complexity','NE_Count','CITY']], on='SiteName', how='left')
)
site_risk['Fault_per_Month']  = site_risk['All_Tickets'] / date_months
site_risk['Low_Vol']          = site_risk['All_Tickets'] < MIN_SITE_TICKETS
site_risk['Complexity_Score'] = site_risk['Complexity'].map({'Low':0,'Medium':50,'High':100})

def pct_rank(s): return s.rank(pct=True, na_option='bottom') * 100
for zone in ZONE_ORDER:
    m = site_risk['ZONE'] == zone
    site_risk.loc[m,'Freq_Rank']   = pct_rank(site_risk.loc[m,'Fault_per_Month'])
    site_risk.loc[m,'MTTR_Rank']   = pct_rank(site_risk.loc[m,'Avg_MTTR'])
    site_risk.loc[m,'Breach_Rank'] = pct_rank(site_risk.loc[m,'Breach_Rate'])

site_risk['Risk_Score'] = (
    site_risk['Freq_Rank'] * 0.30 +
    site_risk['MTTR_Rank'] * 0.30 +
    site_risk['Breach_Rank'] * 0.25 +
    site_risk['Complexity_Score'] * 0.15
)
site_risk = site_risk.sort_values(['ZONE','Risk_Score'], ascending=[True,False])

# Field engineer stats — field groups only
fe_stats = (df_fe.groupby(['ZONE','Field_Lead_ID'])
    .agg(Tickets=('SLA_Compliant','count'),
         SLA_pct=('SLA_Compliant','mean'),
         Sites=('SiteName','nunique'))
    .reset_index())
fe_stats['SLA_pct'] *= 100

reliable      = site_risk[~site_risk['Low_Vol']]
top_risk_site = reliable.nlargest(1,'Risk_Score').iloc[0]
pm_targets    = reliable[reliable['Risk_Score'] >= 80]

print(f"✅ Setup complete")
print(f"   Sites scored (≥{MIN_SITE_TICKETS} tickets): {len(reliable):,}")
print(f"   Field engineers (FO groups only): {df_fe['Field_Lead_ID'].nunique():,}")
if 'WOOwnerGroup' in df_fe_all.columns:
    non_fo = df_fe_all[~df_fe_all['WOOwnerGroup'].isin(FIELD_ENG_GROUPS)]
    print(f"   Non-FO groups in FD path:         {non_fo['Field_Lead_ID'].nunique():,} engineers")
print(f"   PM-priority sites (score ≥80):    {len(pm_targets):,}")
print(f"   Highest-risk site: {top_risk_site['SiteName'].split('_',2)[-1]} "
      f"({top_risk_site['ZONE']} · score {top_risk_site['Risk_Score']:.0f})")

✅ Setup complete
   Sites scored (≥10 tickets): 1,025
   Field engineers (FO groups only): 75
   Non-FO groups in FD path:         154 engineers
   PM-priority sites (score ≥80):    107
   Highest-risk site: d34790a1 (ZONE 3 · score 91)


## 2. Key Metrics Snapshot

In [2]:
TIER_COLORS = {'Low':'#2ca02c','Medium':'#ff7f0e','High':'#d62728'}

# Complexity cross-tab
df_cx = df_zone.merge(site_ne[['SiteName','Complexity']], on='SiteName', how='left')
cx_perf = df_cx.groupby('Complexity').agg(
    Sites=('SiteName','nunique'), Tickets=('SLA_Compliant','count'),
    SLA=('SLA_Compliant','mean'), MTTR=('OUTAGEDURATION','mean')).reset_index()
cx_perf['SLA'] *= 100

# Gini per zone
def gini(arr):
    arr = np.sort(arr.astype(float))
    n = len(arr)
    if n < 2 or arr.sum() == 0: return 0
    return (2 * np.sum(np.arange(1,n+1)*arr) / (n*arr.sum())) - (n+1)/n

worst_gini_zone = max(ZONE_ORDER, key=lambda z:
    gini(fe_stats[fe_stats['ZONE']==z]['Tickets'].values)
    if len(fe_stats[fe_stats['ZONE']==z]) >= 2 else 0)

# Top PM targets: high-risk sites excluding Zone 1 extreme outliers
pm_targets = reliable[reliable['Risk_Score'] >= 80].sort_values('Risk_Score', ascending=False)

metrics = pd.DataFrame({
    'Metric': [
        'Total sites scored (≥10 tickets)',
        'High complexity sites',
        'High complexity — avg MTTR vs NCR avg',
        'Highest-risk site NCR-wide',
        'Zone with most top-risk sites (score ≥80)',
        'Zone 1 extreme MTTR outlier sites',
        'Zone with worst engineer load inequality',
        'PM-priority sites (risk score ≥80)',
    ],
    'Value': [
        f"{len(reliable):,}",
        f"{(site_ne['Complexity']=='High').sum():,} sites "
        f"({(site_ne['Complexity']=='High').mean()*100:.0f}%)",
        f"{cx_perf[cx_perf['Complexity']=='High']['MTTR'].values[0]:.0f}h vs "
        f"{df_zone['OUTAGEDURATION'].mean():.0f}h NCR avg",
        f"{top_risk_site['SiteName'].split('_',2)[-1]}  "
        f"({top_risk_site['ZONE']} · score {top_risk_site['Risk_Score']:.0f})",
        f"{pm_targets['ZONE'].mode()[0] if len(pm_targets) else 'N/A'}  "
        f"({(pm_targets['ZONE']==pm_targets['ZONE'].mode()[0]).sum() if len(pm_targets) else 0} sites)",
        "2 sites with avg MTTR > 200h (see NB16 Section 5)",
        f"{worst_gini_zone}  "
        f"(Gini {gini(fe_stats[fe_stats['ZONE']==worst_gini_zone]['Tickets'].values):.3f})",
        f"{len(pm_targets):,} sites across "
        f"{pm_targets['ZONE'].nunique()} zones",
    ]
})
display(metrics.style
    .set_properties(**{'text-align':'left'})
    .set_table_styles([{'selector':'th','props':[('text-align','left')]}])
    .hide(axis='index')
)

Metric,Value
Total sites scored (≥10 tickets),"1,025"
High complexity sites,537 sites (11%)
High complexity — avg MTTR vs NCR avg,60h vs 58h NCR avg
Highest-risk site NCR-wide,d34790a1 (ZONE 3 · score 91)
Zone with most top-risk sites (score ≥80),ZONE 2 (22 sites)
Zone 1 extreme MTTR outlier sites,2 sites with avg MTTR > 200h (see NB16 Section 5)
Zone with worst engineer load inequality,ZONE 3 (Gini 0.229)
PM-priority sites (risk score ≥80),107 sites across 6 zones


## 3. Key Findings

### Finding 1 🟡 — Site Complexity Correlates with MTTR, Not Strongly with SLA

High-complexity sites (≥3 NE types or Core/IP infrastructure) average higher MTTR than
Low-complexity sites, but SLA compliance differences across tiers are modest (~1pp).
This indicates that high-complexity sites take longer to resolve when faults occur, but
engineers still meet SLA windows at roughly similar rates — suggesting SLA targets are
already calibrated to complexity.

The stronger operational implication: MTTR at High-complexity sites is the primary
contributor to prolonged service disruption, even when the SLA flag is met.
PM targeting at High-complexity sites should prioritise fault prevention over response speed.

### Finding 2 🔴 — Zone 1 MTTR Is Distorted by Two Extreme Outlier Sites

The two highest-volume sites in Zone 1 (Caloocan City) show average MTTR of 433h and 246h —
far exceeding any other site in NCR. These two sites alone significantly inflate Zone 1's
zone-level MTTR average. Area head review required to determine whether this is a chronic
infrastructure problem, an unresolved long-running ticket, or an access/permit issue
requiring capital intervention.

### Finding 3 🟢 — Field Engineer Load Is Well-Distributed Among Active Zone Teams

After applying zone-alignment (home zone = modal zone) and an activity threshold of
≥100 tickets, the 75 active field engineers show Gini coefficients of **0.141–0.229**
across all zones — indicating moderate-to-low load concentration. Workloads are
reasonably balanced: median 203–282 tickets per engineer, maximum 319–547.
No engineers exceed 2× their zone median site count.

| Zone | Engineers | Med tickets | Max tickets | Gini | Med sites | Avg SLA% |
|------|-----------|------------|------------|------|-----------|---------|
| ZONE 1 | 8 | 266 | 407 | 0.206 | 60 | 78.2% |
| ZONE 2 | 15 | 203 | 397 | 0.226 | 58 | 81.3% |
| ZONE 3 | 10 | 282 | 547 | 0.229 | 72 | 79.2% |
| ZONE 4 | 15 | 239 | 319 | 0.141 | 61 | 77.7% |
| ZONE 5 | 19 | 215 | 338 | 0.187 | 50 | 70.4% |
| ZONE 6 | 8 | 246 | 379 | 0.165 | 62 | 72.6% |

> **Note:** The unfiltered pool (all Field_Lead_ID attributed tickets, no zone-alignment)
> produces Gini 0.837–0.890. That figure includes cross-zone assignees and incidental
> contributors with very few tickets — it reflects data scatter, not actual team workload.
> The 75-engineer filtered set is the correct basis for operational assessment.

### Finding 4 🟡 — Non-Field Groups Handle 13.3% of Field Dispatch Tickets

3,035 of 22,822 FD tickets (13.3%) are assigned to WO groups outside Field Operations:
NOC (1,363), Fiber Restoration (820), Core Operations (500), Engineering (209),
Facilities Operations (83), Tier 2 Support (59), Enterprise Operations (1).
NB17 Section 6 breaks this down by zone. Where this is a routing issue, the dispatch
workflow should be corrected; where it is legitimate cross-team support, accountability
should be documented against the responsible team.

### Finding 5 🟡 — ROC Account Has Elevated Unknown RFO Rate

The ROC team account (`ENG_a2fa1599`) logged 23.1% of all NCR tickets with a 29.9%
Unknown RFO rate — vs 2.9% for individual NOC engineers. A targeted ROC workflow
review is warranted: ROC-issued tickets should require the same RFO confirmation step
as NOC-issued tickets.

### Finding 6 🟡 — San Juan City Unknown RFO Driven by Unattributed Engineer IDs

San Juan City: 63.7% Unknown RFO · MTTR 86h · NOC time 38.2h.
Engineer_ID breakdown: Unknown 57.6%, ROC account 10.1%, others <3%.
The primary driver is WO closure without engineer attribution (57.6%), not the ROC
account alone. The ROC account is a secondary contributor. San Juan City is excluded
from city-level RFO recommendations until attribution is resolved.

## 4. Recommendations

### Immediate (0–3 Months)

**REC-P5-01 · Zone 1 — Investigate the Two Extreme MTTR Outlier Sites**
The two Caloocan City sites with 433h and 246h average MTTR are not representative
performance failures — they are specific infrastructure or access problems that remain
unresolved. The area head should personally review recent tickets on these two sites,
confirm RFO, and determine whether the issue is a chronic fault, a lessor dispute, or
a structural equipment failure requiring capital intervention. Until resolved, these
sites should be excluded from Zone 1's operational MTTR targets.

**REC-P5-02 · ROC — Mandate RFO Confirmation on ROC-Issued Tickets**
The ROC account's 29.9% Unknown RFO rate is the highest of any team in NCR.
ROC ticket closure should require the same RFO confirmation step as NOC-issued tickets.
Supervisors should implement a daily review of ROC-closed tickets with Unknown RFO
and require field clarification within 24h of closure.
Target: ROC Unknown RFO rate below 10% within 3 months.

---

### Short-Term (3–6 Months)

**REC-P5-03 · PM Schedule — Top Risk Sites by Zone (NB16 Scorecard)**
107 sites score ≥80 on the composite risk index. Area heads should assign PM visits
for at least the top 10 per zone within the next quarter. High-complexity sites
(Core/IP or ≥3 NE types) in the top-risk tier should receive tandem engineer visits.

**REC-P5-05 · Review FD Tickets Assigned to Non-Field Groups**
3,035 FD tickets (13.3%) are handled by non-Field Operations WO groups.
Where this is a routing issue (wrong WO group assigned), the dispatch workflow should
be corrected. Where it reflects legitimate cross-team support, accountability should
be documented against the responsible team.

---

### Strategic (6–12 Months)

**REC-P5-06 · Proceed to Project 6 — Temporal Patterns & Preventive Maintenance**
With site-level risk profiles established in Project 5, Project 6 will analyse
*when* high-risk sites are most likely to generate faults — mapping Holy Week and
typhoon season surge patterns to specific site categories and complexity tiers.
This enables predictive PM scheduling: prioritise high-risk, high-complexity sites
ahead of known seasonal fault spikes.

---

> **REC-P5-04 — Not Required:** Field engineer load is already well-distributed among
> the 75 active zone engineers (Gini 0.141–0.229, no engineers exceeding 2× zone median
> site count). Site assignment rebalancing is not indicated based on current data.

## 5. Next Steps

| Priority | Action | Owner | Timeline |
|----------|--------|-------|----------|
| 🔴 High | Zone 1 — investigate 2 extreme MTTR outlier sites (NB16 Sec 5) | Zone 1 Area Head | Month 1 |
| 🔴 High | ROC — mandate RFO confirmation on ROC-issued tickets | NOC/ROC Supervisor | Month 1 |
| 🟡 Medium | PM visits — top 10 risk-scored sites per zone (NB16 scorecard) | All Area Heads | Month 2–3 |
| 🟡 Medium | Review FD tickets assigned to non-FO WO groups (NB17 Sec 6) | Ops / NOC Supervisor | Month 2–3 |
| 🟢 Standard | High-complexity PM visits — tandem engineer protocol | Area Heads | Month 3–5 |
| ▶ Next | **Commence Project 6 — Temporal Patterns & Preventive Maintenance** | Analytics | Month 3 |

---

> **Project 6 will answer:** Are the April (Holy Week) and June–September (typhoon season)
> volume surges concentrated in specific complexity tiers or high-risk sites?
> Does PM activity in the weeks before peak season reduce reactive fault incidence?
> Which zones are most exposed to seasonal SLA degradation — and what scheduling
> adjustments would close the gap?